In [6]:
import asyncio

async def say_hello():
    print("Hello")
    await asyncio.sleep(1)
    print("World")
    

In [ ]:
async def main():
    await say_hello()

await main()# asyncio.run(main()) can't be run in jupyter notebook

Hello
World


Task Generation

In [ ]:
async def main():
    task = asyncio.create_task(say_hello())#Create a task and put it into the loop.
    await task
await main()

Hello
World


Future: something gonna happen later

In [ ]:
async def main():
    future = asyncio.Future()
    await future

Where to put async and await:

1. Does the function contain a call to something asynchronous inside it?
→ If yes, that function must be declared async def (it becomes a coroutine).

2. Am I calling something that is itself asynchronous (an "awaitable" — a coroutine, a Task, or similar)?

Examples:
1. time.sleep(1) -> await asyncio.sleep(1)

Eg for comparison between the same algo with and without async

In [ ]:
import time
import requests

def fetch_url(url):
    """模拟一个耗时的网络请求（同步版本）"""
    print(f"开始获取: {url}")
    time.sleep(2)  # 模拟 2 秒网络延迟
    print(f"完成获取: {url}")
    return f"来自 {url} 的数据"

def main_sync():
    urls = ['https://example.com/1', 'https://example.com/2', 'https://example.com/3']
    results = []
    start = time.time()
    
    for url in urls:
        result = fetch_url(url)  # 必须等上一个完成才能开始下一个
        results.append(result)
    
    end = time.time()
    print(f"同步版本总耗时: {end - start:.2f} 秒")
    print(f"结果: {results}")

if __name__ == "__main__":
    main_sync()

In [13]:
import asyncio
import aiohttp
import time

async def fetch_url_async(session, url):
    """模拟一个耗时的网络请求（异步版本）"""
    print(f"开始异步获取: {url}")
    # 注意：这里我们使用 aiohttp 的异步 get 方法，并用 await 等待
    async with session.get(url) as response:
        # 模拟处理响应也需要时间
        await asyncio.sleep(2)  # 使用 asyncio.sleep 模拟 I/O 等待，它不会阻塞线程
        text = await response.text()
        print(f"完成异步获取: {url}")
        return f"来自 {url} 的数据 (长度: {len(text)})"

async def main_async():
    urls = ['https://httpbin.org/get', 'https://httpbin.org/delay/1', 'https://httpbin.org/headers']
    
    async with aiohttp.ClientSession() as session:  # 创建异步 HTTP 会话
        # 为每个 URL 创建一个任务（Task）
        tasks = []
        for url in urls:
            # create_task 会将协程加入事件循环，立即开始调度
            task = asyncio.create_task(fetch_url_async(session, url))
            tasks.append(task)
        
        print("所有任务已创建，开始并发执行...")
        
        # 使用 asyncio.gather 并发运行所有任务，并等待它们全部完成
        # gather 返回一个结果列表，顺序与传入的任务顺序一致
        results = await asyncio.gather(*tasks)
        
        return results

if __name__ == "__main__":
    start = time.time()
    # asyncio.run() 是启动事件循环并运行顶层协程的简便方法
    final_results = await main_async()
    end = time.time()
    
    print(f"\n异步版本总耗时: {end - start:.2f} 秒")
    for res in final_results:
        print(res)

所有任务已创建，开始并发执行...
开始异步获取: https://httpbin.org/get
开始异步获取: https://httpbin.org/delay/1
开始异步获取: https://httpbin.org/headers
完成异步获取: https://httpbin.org/headers
完成异步获取: https://httpbin.org/get
完成异步获取: https://httpbin.org/delay/1

异步版本总耗时: 3.35 秒
来自 https://httpbin.org/get 的数据 (长度: 311)
来自 https://httpbin.org/delay/1 的数据 (长度: 361)
来自 https://httpbin.org/headers 的数据 (长度: 229)


# concurrently solving multi-tasks

## Without async

In [18]:
import time

def task1():
    print("Task 1 started")
    time.sleep(1)
    print("Task 1 finished")

def task2():
    print("Task 2 started")
    time.sleep(2)

    print("Task 2 finished")

def main():
    start = time.time()
    task1()
    task2()
    end = time.time()
    print(end - start)

main()

Task 1 started
Task 1 finished
Task 2 started
Task 2 finished
3.0057389736175537


## With async

In [16]:
import asyncio

async def task1():
    print("Task 1 started")
    await asyncio.sleep(1)
    print("Task 1 finished")

async def task2():
    print("Task 2 started")
    await asyncio.sleep(2)
    print("Task 2 finished")

async def main():
    start = time.time()

    await asyncio.gather(task1(), task2())
    end = time.time()
    print(end - start)

await main()

Task 1 started
Task 2 started
Task 1 finished
Task 2 finished
2.001641035079956


# Out-of-Time control

In [ ]:
import asyncio

async def long_task():
    await asyncio.sleep(5)
    print("Task finished")

async def main():
    try:
        await asyncio.wait_for(long_task(), timeout=5)# time < 5
    except asyncio.TimeoutError:
        print("Task timed out")

await main()

Task timed out


In [ ]:
import asyncio
async def producer(queue):
    for i in range(5):
        await queue.put(i)
        await asyncio.sleep(0.1)

async def consumer(queue):
    while True:
        item = await queue.get()
        if item is None:      # sentinel received, exit the loop
            queue.task_done()
            break
        print(f"Consumed {item}")
        queue.task_done()

async def main():
    queue = asyncio.Queue()
    await asyncio.gather(
        producer(queue),
        consumer(queue)
    )
await main()

Consumed 0
Consumed 1
Consumed 2
Consumed 3
Consumed 4
